In [1]:
import numpy as np
import random
from scipy.stats import truncnorm
import math
import time
from scipy.stats import ttest_rel

#Gera população inicial

In [2]:
def gera_individuo(n):
  #limites do intervalo desejado
  limite_inferior = -5.12
  limite_superior = 5.12

  #parâmetros da curva normal original
  media = 0.0
  desvio_padrao = 1# Um desvio menor concentra mais os dados no centro
  quantidade = n

  #ajuste de limites para a escala da função truncnorm
  a = (limite_inferior - media) / desvio_padrao
  b = (limite_superior - media) / desvio_padrao

  #gerar os valores aleatórios
  valores = truncnorm.rvs(a, b, loc=media, scale=desvio_padrao, size=quantidade)

  return valores

def gerar_populacao(m, n):
    #Vamos considerar uma população de tamanho m com uma fita de dna de tamanho n
    populacao = np.zeros((m,n))
    for i in range(m):
        populacao[i] = gera_individuo(n)
    return populacao

#Função objetivo

In [3]:
#Aqui é realizado o calculo numérico da função objetivo
def fo(x):
    return 10 * len(x) + np.sum(x ** 2 - 10 * np.cos(2 * np.pi * x))

#Mutação diferencial

In [4]:
def mutacao_diferencial(x1, x2, x3, F=0.5):
    return x1 + F * (x2 - x3)

def crossover(x, v, CR):
    n = len(x)
    u = np.copy(x)

    j_rand = np.random.randint(n)

    for j in range(n):
        if np.random.uniform() <= CR or j == j_rand:
            u[j] = v[j]

    return u


#Configuração do DE

In [5]:
def de(caso_teste, semente):
    Np = 100
    n = 100
    CR = 0.9

    np.random.seed(semente)
    random.seed(semente)

    pop = gerar_populacao(Np, n)
    fitness = np.zeros(Np)

    #inicio da contagem de tempo
    inicio = time.perf_counter()

    # Avaliação inicial
    for i in range(Np):
        fitness[i] = fo(pop[i])
    for i in range(300):

        nova_pop = np.copy(pop)
        novo_fitness = np.copy(fitness)

        for j in range(Np):

            # Seleciona os 3 elementos da mutação
            indices = [k for k in range(Np) if k != j]
            x1, x2, x3 = random.sample(indices, 3)

            # Mutação diferencial
            if caso_teste == "C1":
              F = 0.5
              v = mutacao_diferencial(pop[x1], pop[x2], pop[x3], F)
            else:
              F = 0.9
              v = mutacao_diferencial(pop[x1], pop[x2], pop[x3], F)

            # Crossover
            u = crossover(pop[j], v, CR)

            # Tratamento dos limites
            u = np.clip(u, -5.12, 5.12)

            # Seleção
            f_u = fo(u)

            if f_u <= fitness[j]:
                nova_pop[j] = u
                novo_fitness[j] = f_u

        pop = nova_pop
        fitness = novo_fitness

    melhor = np.argmin(fitness)
    pior = np.argmax(fitness)

    # Finaliza a contagem
    fim = time.perf_counter()
    tempo_execucao = fim - inicio

    return pop[melhor], tempo_execucao, fitness[melhor], pop[pior], fitness[pior]

#main

In [6]:
numero_execucoes = 30
alpha = 0.05 #Tolerancia para aceitar ou regeitar a hipótese nula (H0)

# Resultados de cada caso de teste
resultados_C1 = []
resultados_C2 = []

tempos_C1 = []
tempos_C2 = []

# Listas para armazenar as melhores e piores soluções
melhores_solucoes_C1 = []
piores_solucoes_C1 = []

melhores_solucoes_C2 = []
piores_solucoes_C2 = []

# Fitness das melhores e piores soluções
melhores_fitness_C1 = []
piores_fitness_C1 = []

melhores_fitness_C2 = []
piores_fitness_C2 = []

# Executa os dois casos com sementes correspondentes
for execucao in range(numero_execucoes):

    # A mesma semente será utilizada nos dois casos
    semente = 1000 + execucao

    for caso_teste in ["C1", "C2"]:

        s_star, tempo, f_star, s_pior, f_pior = de(caso_teste, semente)

        if caso_teste == "C1":
            resultados_C1.append(f_star)
            tempos_C1.append(tempo)

            melhores_solucoes_C1.append(s_star.copy())
            piores_solucoes_C1.append(s_pior.copy())

            melhores_fitness_C1.append(f_star)
            piores_fitness_C1.append(f_pior)

        else:
            resultados_C2.append(f_star)
            tempos_C2.append(tempo)

            melhores_solucoes_C2.append(s_star.copy())
            piores_solucoes_C2.append(s_pior.copy())

            melhores_fitness_C2.append(f_star)
            piores_fitness_C2.append(f_pior)

        print(f"Execução {execucao + 1:02d}/30 | {caso_teste} | Fitness: {f_star:.8f} | Tempo: {tempo:.2f}s")


# Converte os resultados em arrays
resultados_C1 = np.array(resultados_C1)
resultados_C2 = np.array(resultados_C2)

tempos_C1 = np.array(tempos_C1)
tempos_C2 = np.array(tempos_C2)


# Estatísticas descritivas
print("\n" + "=" * 50)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 50)

print("\nC1 - F = 0,5")
print(f"Fitness médio: {np.mean(resultados_C1):.8f} +/- {np.std(resultados_C1, ddof=1):.8f}")
print(f"Tempo médio: {np.mean(tempos_C1):.4f} +/- {np.std(tempos_C1, ddof=1):.4f} segundos")

print("\nC2 - F = 0,9")
print(f"Fitness médio: {np.mean(resultados_C2):.8f} +/- {np.std(resultados_C2, ddof=1):.8f}")
print(f"Tempo médio: {np.mean(tempos_C2):.4f} +/- {np.std(tempos_C2, ddof=1):.4f} segundos")


# Teste t pareado para o fitness
t_fitness, p_fitness = ttest_rel(resultados_C1, resultados_C2)

# Teste t pareado para o tempo
t_tempo, p_tempo = ttest_rel(tempos_C1, tempos_C2)


# Resultados dos testes estatísticos
print("\n" + "=" * 50)
print("TESTE T PAREADO")
print("=" * 50)

print("\nFitness")
print(f"Estatística t: {t_fitness:.6f}")
print(f"p-valor: {p_fitness:.8f}")

if p_fitness < alpha:
    print("Conclusão: rejeita-se H0. Há evidências de diferença entre as médias do fitness.")
else:
    print("Conclusão: não se rejeita H0. Não há evidências suficientes de diferença entre as médias.")

print("\nTempo de execução")
print(f"Estatística t: {t_tempo:.6f}")
print(f"p-valor: {p_tempo:.8f}")

if p_tempo < alpha:
    print("Conclusão: rejeita-se H0. Há evidências de diferença entre as médias dos tempos.")
else:
    print("Conclusão: não se rejeita H0. Não há evidências suficientes de diferença entre as médias dos tempos.")


# ==========================================
# MELHORES E PIORES SOLUÇÕES GLOBAIS
# ==========================================

# C1
idx_melhor_C1 = np.argmin(melhores_fitness_C1)
idx_pior_C1 = np.argmax(piores_fitness_C1)

melhor_fitness_global_C1 = melhores_fitness_C1[idx_melhor_C1]
melhor_solucao_global_C1 = melhores_solucoes_C1[idx_melhor_C1]

pior_fitness_global_C1 = piores_fitness_C1[idx_pior_C1]
pior_solucao_global_C1 = piores_solucoes_C1[idx_pior_C1]


# C2
idx_melhor_C2 = np.argmin(melhores_fitness_C2)
idx_pior_C2 = np.argmax(piores_fitness_C2)

melhor_fitness_global_C2 = melhores_fitness_C2[idx_melhor_C2]
melhor_solucao_global_C2 = melhores_solucoes_C2[idx_melhor_C2]

pior_fitness_global_C2 = piores_fitness_C2[idx_pior_C2]
pior_solucao_global_C2 = piores_solucoes_C2[idx_pior_C2]


# Exibição dos resultados globais
print("\n" + "=" * 60)
print("MELHORES E PIORES SOLUÇÕES GLOBAIS")
print("=" * 60)

print("\nC1 - F = 0,5")
print(f"Melhor fitness: {melhor_fitness_global_C1:.8f}")
print(f"Execução da melhor solução: {idx_melhor_C1 + 1}")
print(f"Melhor solução: {melhor_solucao_global_C1}")

print(f"\nPior fitness: {pior_fitness_global_C1:.8f}")
print(f"Execução da pior solução: {idx_pior_C1 + 1}")
print(f"Pior solução: {pior_solucao_global_C1}")


print("\nC2 - F = 0,9")
print(f"Melhor fitness: {melhor_fitness_global_C2:.8f}")
print(f"Execução da melhor solução: {idx_melhor_C2 + 1}")
print(f"Melhor solução: {melhor_solucao_global_C2}")

print(f"\nPior fitness: {pior_fitness_global_C2:.8f}")
print(f"Execução da pior solução: {idx_pior_C2 + 1}")
print(f"Pior solução: {pior_solucao_global_C2}")


# ==========================================
# SALVAMENTO DAS SOLUÇÕES EM ARQUIVOS TXT
# ==========================================

# C1
np.savetxt("melhor_solucao_C1.txt", melhor_solucao_global_C1, fmt="%.10f")
np.savetxt("pior_solucao_C1.txt", pior_solucao_global_C1, fmt="%.10f")

# C2
np.savetxt("melhor_solucao_C2.txt", melhor_solucao_global_C2, fmt="%.10f")
np.savetxt("pior_solucao_C2.txt", pior_solucao_global_C2, fmt="%.10f")


# ==========================================
# SALVAMENTO DOS RESULTADOS DAS EXECUÇÕES
# ==========================================

# Salva os melhores fitness e tempos de cada execução
np.savetxt("resultados_C1.txt", np.column_stack((resultados_C1, tempos_C1)), fmt=["%.10f", "%.6f"], header="Fitness Tempo")

np.savetxt("resultados_C2.txt", np.column_stack((resultados_C2, tempos_C2)), fmt=["%.10f", "%.6f"], header="Fitness Tempo")

Execução 01/30 | C1 | Fitness: 861.60989580 | Tempo: 18.82s
Execução 01/30 | C1 | Fitness: 861.60989580 | Tempo: 18.82s
Execução 01/30 | C2 | Fitness: 945.18278259 | Tempo: 12.51s
Execução 01/30 | C2 | Fitness: 945.18278259 | Tempo: 12.51s
Execução 02/30 | C1 | Fitness: 872.98105145 | Tempo: 12.52s
Execução 02/30 | C1 | Fitness: 872.98105145 | Tempo: 12.52s
Execução 02/30 | C2 | Fitness: 923.34710784 | Tempo: 12.49s
Execução 02/30 | C2 | Fitness: 923.34710784 | Tempo: 12.49s
Execução 03/30 | C1 | Fitness: 847.26687679 | Tempo: 13.20s
Execução 03/30 | C1 | Fitness: 847.26687679 | Tempo: 13.20s
Execução 03/30 | C2 | Fitness: 883.49604762 | Tempo: 12.63s
Execução 03/30 | C2 | Fitness: 883.49604762 | Tempo: 12.63s
Execução 04/30 | C1 | Fitness: 850.52665307 | Tempo: 12.45s
Execução 04/30 | C1 | Fitness: 850.52665307 | Tempo: 12.45s
Execução 04/30 | C2 | Fitness: 934.00553546 | Tempo: 11.57s
Execução 04/30 | C2 | Fitness: 934.00553546 | Tempo: 11.57s
Execução 05/30 | C1 | Fitness: 806.29033